# Use Case 02 — Real-Time Fraud Detection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/usecases/notebooks/02-fraud-detection.ipynb)

**CareerAlign — GCP Professional Machine Learning Engineer**

Build a complete fraud detection pipeline: synthetic data generation, feature engineering, class imbalance handling, XGBoost training, evaluation, and threshold optimization.

---

## 0 — Setup & Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost matplotlib seaborn imbalanced-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score, roc_curve,
    f1_score, fbeta_score
)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Plot style
plt.style.use('dark_background')
sns.set_palette('Set2')
print('All libraries loaded successfully.')

---
## 1 — Generate Synthetic Transaction Data

We create 10,000 transactions with a realistic 0.5% fraud rate. Fraudulent transactions exhibit different patterns: unusual amounts, odd hours, high velocity, geographic anomalies.

In [ ]:
n_transactions = 10_000
fraud_rate = 0.005  # 0.5%
n_fraud = int(n_transactions * fraud_rate)
n_legit = n_transactions - n_fraud

print(f"Generating {n_transactions:,} transactions")
print(f"  Legitimate: {n_legit:,} ({100*(1-fraud_rate):.1f}%)")
print(f"  Fraudulent: {n_fraud:,} ({100*fraud_rate:.1f}%)")

In [ ]:
def generate_transactions(n_legit, n_fraud, seed=42):
    """Generate synthetic credit card transaction data."""
    rng = np.random.RandomState(seed)

    # --- Legitimate transactions ---
    legit = pd.DataFrame({
        'amount': rng.lognormal(mean=3.5, sigma=1.0, size=n_legit).clip(0.5, 5000),
        'hour_of_day': rng.choice(range(24), size=n_legit, p=[
            0.01, 0.005, 0.005, 0.005, 0.005, 0.01,  # 0-5 (low activity)
            0.03, 0.05, 0.07, 0.08, 0.08, 0.08,       # 6-11 (morning)
            0.08, 0.07, 0.06, 0.05, 0.05, 0.06,       # 12-17 (afternoon)
            0.06, 0.05, 0.04, 0.03, 0.02, 0.015       # 18-23 (evening)
        ]),
        'day_of_week': rng.choice(range(7), size=n_legit,
                                   p=[0.16, 0.16, 0.15, 0.14, 0.14, 0.13, 0.12]),
        'merchant_category': rng.choice(
            ['grocery', 'restaurant', 'gas', 'retail', 'online', 'travel', 'entertainment'],
            size=n_legit, p=[0.25, 0.20, 0.15, 0.15, 0.12, 0.08, 0.05]
        ),
        'is_international': rng.binomial(1, 0.05, size=n_legit),
        'distance_from_home_km': rng.exponential(10, size=n_legit).clip(0, 500),
        'time_since_last_txn_min': rng.exponential(120, size=n_legit).clip(1, 10000),
        'is_weekend': 0,
        'is_night': 0,
        'device_change': rng.binomial(1, 0.02, size=n_legit),
        'num_txn_last_1h': rng.poisson(0.5, size=n_legit),
        'num_txn_last_24h': rng.poisson(3, size=n_legit),
        'avg_amount_30d': rng.lognormal(mean=3.5, sigma=0.5, size=n_legit).clip(10, 2000),
        'is_fraud': 0,
    })
    legit['is_weekend'] = (legit['day_of_week'] >= 5).astype(int)
    legit['is_night'] = ((legit['hour_of_day'] >= 23) | (legit['hour_of_day'] <= 5)).astype(int)

    # --- Fraudulent transactions ---
    fraud = pd.DataFrame({
        'amount': rng.lognormal(mean=5.0, sigma=1.5, size=n_fraud).clip(50, 15000),
        'hour_of_day': rng.choice(range(24), size=n_fraud, p=[
            0.06, 0.06, 0.07, 0.07, 0.06, 0.05,  # 0-5 (higher nighttime)
            0.03, 0.03, 0.03, 0.04, 0.04, 0.04,   # 6-11
            0.04, 0.04, 0.04, 0.04, 0.04, 0.04,   # 12-17
            0.04, 0.04, 0.03, 0.03, 0.04, 0.04    # 18-23
        ]),
        'day_of_week': rng.choice(range(7), size=n_fraud),
        'merchant_category': rng.choice(
            ['grocery', 'restaurant', 'gas', 'retail', 'online', 'travel', 'entertainment'],
            size=n_fraud, p=[0.05, 0.05, 0.05, 0.10, 0.40, 0.25, 0.10]
        ),
        'is_international': rng.binomial(1, 0.45, size=n_fraud),
        'distance_from_home_km': rng.exponential(200, size=n_fraud).clip(50, 10000),
        'time_since_last_txn_min': rng.exponential(10, size=n_fraud).clip(0.1, 500),
        'is_weekend': 0,
        'is_night': 0,
        'device_change': rng.binomial(1, 0.6, size=n_fraud),
        'num_txn_last_1h': rng.poisson(4, size=n_fraud),
        'num_txn_last_24h': rng.poisson(12, size=n_fraud),
        'avg_amount_30d': rng.lognormal(mean=3.5, sigma=0.5, size=n_fraud).clip(10, 2000),
        'is_fraud': 1,
    })
    fraud['is_weekend'] = (fraud['day_of_week'] >= 5).astype(int)
    fraud['is_night'] = ((fraud['hour_of_day'] >= 23) | (fraud['hour_of_day'] <= 5)).astype(int)

    df = pd.concat([legit, fraud], ignore_index=True)
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    return df

df = generate_transactions(n_legit, n_fraud)
print(f"Dataset shape: {df.shape}")
print(f"\nFraud distribution:")
print(df['is_fraud'].value_counts())
df.head()

---
## 2 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Fraud vs Legitimate Transaction Distributions', fontsize=16, y=1.02)

# 1. Amount distribution
ax = axes[0, 0]
for label, color in [(0, '#22d3ee'), (1, '#ef4444')]:
    subset = df[df['is_fraud'] == label]['amount']
    ax.hist(subset, bins=50, alpha=0.6, color=color,
            label='Legit' if label == 0 else 'Fraud', density=True)
ax.set_xlabel('Amount ($)')
ax.set_ylabel('Density')
ax.set_title('Transaction Amount')
ax.legend()
ax.set_xlim(0, 3000)

# 2. Hour of day
ax = axes[0, 1]
for label, color in [(0, '#22d3ee'), (1, '#ef4444')]:
    subset = df[df['is_fraud'] == label]['hour_of_day']
    ax.hist(subset, bins=24, alpha=0.6, color=color,
            label='Legit' if label == 0 else 'Fraud', density=True)
ax.set_xlabel('Hour of Day')
ax.set_title('Transaction Time')
ax.legend()

# 3. Distance from home
ax = axes[0, 2]
for label, color in [(0, '#22d3ee'), (1, '#ef4444')]:
    subset = df[df['is_fraud'] == label]['distance_from_home_km']
    ax.hist(subset, bins=50, alpha=0.6, color=color,
            label='Legit' if label == 0 else 'Fraud', density=True)
ax.set_xlabel('Distance from Home (km)')
ax.set_title('Geographic Distance')
ax.legend()
ax.set_xlim(0, 1000)

# 4. Transactions in last hour
ax = axes[1, 0]
for label, color in [(0, '#22d3ee'), (1, '#ef4444')]:
    subset = df[df['is_fraud'] == label]['num_txn_last_1h']
    ax.hist(subset, bins=15, alpha=0.6, color=color,
            label='Legit' if label == 0 else 'Fraud', density=True)
ax.set_xlabel('# Transactions in Last Hour')
ax.set_title('Transaction Velocity (1h)')
ax.legend()

# 5. Merchant category
ax = axes[1, 1]
cat_fraud = df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False)
cat_fraud.plot(kind='bar', ax=ax, color='#f59e0b', alpha=0.8)
ax.set_ylabel('Fraud Rate')
ax.set_title('Fraud Rate by Merchant Category')
ax.tick_params(axis='x', rotation=45)

# 6. International vs domestic
ax = axes[1, 2]
intl_fraud = df.groupby('is_international')['is_fraud'].mean()
intl_fraud.index = ['Domestic', 'International']
intl_fraud.plot(kind='bar', ax=ax, color=['#22d3ee', '#ef4444'], alpha=0.8)
ax.set_ylabel('Fraud Rate')
ax.set_title('Fraud Rate: Domestic vs International')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---
## 3 — Feature Engineering

Create derived features that capture behavioral anomalies — the key signal for fraud detection.

In [ ]:
def engineer_features(df):
    """Create fraud-specific derived features."""
    df = df.copy()

    # Amount-based features
    df['amount_to_avg_ratio'] = df['amount'] / df['avg_amount_30d'].clip(lower=1)
    df['amount_log'] = np.log1p(df['amount'])
    df['is_high_amount'] = (df['amount'] > df['avg_amount_30d'] * 3).astype(int)
    df['amount_zscore'] = (df['amount'] - df['avg_amount_30d']) / df['avg_amount_30d'].clip(lower=1)

    # Velocity features
    df['velocity_1h_to_24h_ratio'] = (
        df['num_txn_last_1h'] / df['num_txn_last_24h'].clip(lower=1)
    )
    df['is_rapid_succession'] = (df['time_since_last_txn_min'] < 5).astype(int)
    df['txn_frequency_score'] = df['num_txn_last_1h'] * 24 / df['num_txn_last_24h'].clip(lower=1)

    # Time-based features
    df['is_unusual_hour'] = (
        (df['hour_of_day'] >= 0) & (df['hour_of_day'] <= 5)
    ).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour_of_day'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour_of_day'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # Geographic features
    df['distance_log'] = np.log1p(df['distance_from_home_km'])
    df['is_far_from_home'] = (df['distance_from_home_km'] > 200).astype(int)

    # Interaction features
    df['night_and_international'] = df['is_night'] * df['is_international']
    df['high_amount_and_device_change'] = df['is_high_amount'] * df['device_change']
    df['rapid_and_international'] = df['is_rapid_succession'] * df['is_international']
    df['night_and_high_velocity'] = df['is_night'] * (df['num_txn_last_1h'] > 3).astype(int)

    # One-hot encode merchant category
    df = pd.get_dummies(df, columns=['merchant_category'], prefix='mcc', drop_first=True)

    return df

df_feat = engineer_features(df)
print(f"Features before engineering: {df.shape[1]}")
print(f"Features after engineering:  {df_feat.shape[1]}")
print(f"\nNew features:")
new_cols = [c for c in df_feat.columns if c not in df.columns]
for col in new_cols:
    print(f"  - {col}")

---
## 4 — Train/Test Split

In [ ]:
# Separate features and target
target = 'is_fraud'
drop_cols = [target]

feature_cols = [c for c in df_feat.columns if c not in drop_cols]
X = df_feat[feature_cols].values
y = df_feat[target].values

# Stratified split to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)

print(f"Train: {X_train.shape[0]:,} samples ({sum(y_train)} fraud = {100*sum(y_train)/len(y_train):.2f}%)")
print(f"Val:   {X_val.shape[0]:,} samples ({sum(y_val)} fraud = {100*sum(y_val)/len(y_val):.2f}%)")
print(f"Test:  {X_test.shape[0]:,} samples ({sum(y_test)} fraud = {100*sum(y_test)/len(y_test):.2f}%)")
print(f"\nFeatures: {X_train.shape[1]}")

---
## 5 — Handling Class Imbalance

We compare three strategies:
1. **No treatment** (baseline)
2. **SMOTE** (synthetic oversampling of minority class)
3. **Class weights** (penalize minority misclassification)

In [ ]:
# Strategy 1: No resampling (baseline)
X_train_base, y_train_base = X_train, y_train
print(f"Baseline — Fraud: {sum(y_train_base)}, Legit: {len(y_train_base) - sum(y_train_base)}")

# Strategy 2: SMOTE
smote = SMOTE(sampling_strategy=0.3, k_neighbors=5, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"SMOTE    — Fraud: {sum(y_train_smote)}, Legit: {len(y_train_smote) - sum(y_train_smote)}")

# Strategy 3: SMOTE + Tomek links (clean boundary)
smote_tomek = SMOTETomek(
    smote=SMOTE(sampling_strategy=0.3, random_state=42),
    random_state=42
)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train, y_train)
print(f"SMOTE+T  — Fraud: {sum(y_train_st)}, Legit: {len(y_train_st) - sum(y_train_st)}")

# Class weight for XGBoost
scale_pos_weight = sum(y_train == 0) / max(sum(y_train == 1), 1)
print(f"\nscale_pos_weight for XGBoost: {scale_pos_weight:.1f}")

---
## 6 — XGBoost Training & Comparison

Train multiple models to compare imbalance-handling strategies.

In [ ]:
def train_xgb(X_tr, y_tr, X_v, y_v, scale_pos_weight=1, label='model'):
    """Train an XGBoost classifier and return model + validation predictions."""
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        max_depth=6,
        learning_rate=0.05,
        n_estimators=500,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight,
        tree_method='hist',
        random_state=42,
        use_label_encoder=False,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_v, y_v)],
        verbose=0,
    )
    y_prob = model.predict_proba(X_v)[:, 1]
    pr_auc = average_precision_score(y_v, y_prob)
    roc = roc_auc_score(y_v, y_prob)
    print(f"  {label:20s} — PR-AUC: {pr_auc:.4f} | ROC-AUC: {roc:.4f}")
    return model, y_prob

print("Training models...\n")

# Model A: Baseline (no imbalance handling)
model_base, prob_base = train_xgb(
    X_train_base, y_train_base, X_val, y_val,
    scale_pos_weight=1, label='Baseline'
)

# Model B: Class weights
model_weighted, prob_weighted = train_xgb(
    X_train, y_train, X_val, y_val,
    scale_pos_weight=scale_pos_weight, label='Class Weights'
)

# Model C: SMOTE
model_smote, prob_smote = train_xgb(
    X_train_smote, y_train_smote, X_val, y_val,
    scale_pos_weight=1, label='SMOTE'
)

# Model D: SMOTE + Tomek + Class weights
model_combo, prob_combo = train_xgb(
    X_train_st, y_train_st, X_val, y_val,
    scale_pos_weight=3, label='SMOTE+Tomek+Weights'
)

---
## 7 — Precision-Recall Curves

PR-AUC is the preferred metric for imbalanced classification. ROC-AUC can be misleadingly high.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_info = [
    ('Baseline', prob_base, '#8e94b5'),
    ('Class Weights', prob_weighted, '#22d3ee'),
    ('SMOTE', prob_smote, '#22c55e'),
    ('SMOTE+Tomek+Wt', prob_combo, '#f59e0b'),
]

# Precision-Recall curves
ax = axes[0]
for name, probs, color in models_info:
    precision, recall, _ = precision_recall_curve(y_val, probs)
    ap = average_precision_score(y_val, probs)
    ax.plot(recall, precision, color=color, lw=2, label=f'{name} (AP={ap:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=14)
ax.legend(loc='best', fontsize=10)
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.2)

# ROC curves
ax = axes[1]
for name, probs, color in models_info:
    fpr, tpr, _ = roc_curve(y_val, probs)
    auc = roc_auc_score(y_val, probs)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'w--', alpha=0.3, lw=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("\nNote: PR-AUC differences are more meaningful than ROC-AUC for imbalanced data.")

---
## 8 — Select Best Model & Evaluate on Test Set

In [ ]:
# Select the best model (Class Weights typically works well)
best_model = model_weighted
best_label = 'Class Weights'

# Predict on test set
y_test_prob = best_model.predict_proba(X_test)[:, 1]

# Default threshold = 0.5
y_test_pred = (y_test_prob >= 0.5).astype(int)

print(f"Best model: {best_label}")
print(f"\n{'='*50}")
print("TEST SET RESULTS (threshold=0.5)")
print(f"{'='*50}")
print(f"\nROC-AUC:  {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"PR-AUC:   {average_precision_score(y_test, y_test_prob):.4f}")
print(f"F1-Score: {f1_score(y_test, y_test_pred):.4f}")
print(f"F2-Score: {fbeta_score(y_test, y_test_pred, beta=2):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Legit', 'Fraud']))

---
## 9 — Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrBr', ax=axes[0],
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized
cm_norm = confusion_matrix(y_test, y_test_pred, normalize='true')
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='YlOrBr', ax=axes[1],
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Normalized)')

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives:  {tn:,} (legit correctly approved)")
print(f"False Positives: {fp:,} (legit incorrectly blocked)")
print(f"False Negatives: {fn:,} (fraud missed — COSTLY)")
print(f"True Positives:  {tp:,} (fraud correctly caught)")

---
## 10 — Feature Importance Analysis

In [ ]:
# Get feature importances
importances = best_model.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

# Plot top 15 features
fig, ax = plt.subplots(figsize=(10, 8))
top_n = 15
top_feats = importance_df.head(top_n)

bars = ax.barh(
    range(top_n), top_feats['importance'].values,
    color='#f59e0b', alpha=0.85, edgecolor='#f59e0b'
)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_feats['feature'].values, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gain)', fontsize=12)
ax.set_title('Top 15 Features for Fraud Detection', fontsize=14)
ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.show()

print("\nTop 10 features:")
for i, row in importance_df.head(10).iterrows():
    print(f"  {row['feature']:35s} {row['importance']:.4f}")

---
## 11 — Threshold Optimization

The default threshold of 0.5 is rarely optimal for fraud detection. We optimize using a **cost-sensitive** approach:
- Cost of missed fraud (FN): $500 average
- Cost of false alarm (FP): $15 (lost fee + customer friction)
- Cost of investigation (TP): $3

In [ ]:
# Cost parameters
COST_FN = 500   # missed fraud
COST_FP = 15    # false positive (blocked legit)
COST_TP = 3     # investigation cost
COST_TN = 0     # correct approval

thresholds = np.arange(0.05, 0.95, 0.01)
results = []

for thresh in thresholds:
    y_pred_t = (y_test_prob >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    total_cost = (fn * COST_FN) + (fp * COST_FP) + (tp * COST_TP) + (tn * COST_TN)
    recall = tp / max(tp + fn, 1)
    precision = tp / max(tp + fp, 1)
    f2 = fbeta_score(y_test, y_pred_t, beta=2) if (tp + fp + fn) > 0 else 0

    results.append({
        'threshold': thresh,
        'total_cost': total_cost,
        'recall': recall,
        'precision': precision,
        'f2_score': f2,
        'fp_count': fp,
        'fn_count': fn,
    })

results_df = pd.DataFrame(results)

# Find optimal thresholds
opt_cost_idx = results_df['total_cost'].idxmin()
opt_f2_idx = results_df['f2_score'].idxmax()

print(f"Optimal threshold (min cost):  {results_df.loc[opt_cost_idx, 'threshold']:.2f}")
print(f"  Total cost: ${results_df.loc[opt_cost_idx, 'total_cost']:,.0f}")
print(f"  Recall: {results_df.loc[opt_cost_idx, 'recall']:.3f}")
print(f"  Precision: {results_df.loc[opt_cost_idx, 'precision']:.3f}")
print()
print(f"Optimal threshold (max F2):    {results_df.loc[opt_f2_idx, 'threshold']:.2f}")
print(f"  F2-Score: {results_df.loc[opt_f2_idx, 'f2_score']:.3f}")
print(f"  Recall: {results_df.loc[opt_f2_idx, 'recall']:.3f}")
print(f"  Precision: {results_df.loc[opt_f2_idx, 'precision']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Cost vs threshold
ax = axes[0]
ax.plot(results_df['threshold'], results_df['total_cost'], color='#ef4444', lw=2)
opt_thresh = results_df.loc[opt_cost_idx, 'threshold']
opt_cost = results_df.loc[opt_cost_idx, 'total_cost']
ax.axvline(opt_thresh, color='#f59e0b', ls='--', alpha=0.8, label=f'Optimal: {opt_thresh:.2f}')
ax.scatter([opt_thresh], [opt_cost], color='#f59e0b', s=100, zorder=5)
ax.set_xlabel('Threshold')
ax.set_ylabel('Total Cost ($)')
ax.set_title('Cost-Sensitive Threshold')
ax.legend()
ax.grid(alpha=0.2)

# 2. Precision & Recall vs threshold
ax = axes[1]
ax.plot(results_df['threshold'], results_df['recall'], color='#22c55e', lw=2, label='Recall')
ax.plot(results_df['threshold'], results_df['precision'], color='#6366f1', lw=2, label='Precision')
ax.plot(results_df['threshold'], results_df['f2_score'], color='#f59e0b', lw=2, label='F2-Score')
ax.axvline(opt_thresh, color='#f59e0b', ls='--', alpha=0.5)
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F2 vs Threshold')
ax.legend()
ax.grid(alpha=0.2)

# 3. FP and FN counts
ax = axes[2]
ax.plot(results_df['threshold'], results_df['fp_count'], color='#22d3ee', lw=2, label='False Positives')
ax.plot(results_df['threshold'], results_df['fn_count'], color='#ef4444', lw=2, label='False Negatives')
ax.axvline(opt_thresh, color='#f59e0b', ls='--', alpha=0.5)
ax.set_xlabel('Threshold')
ax.set_ylabel('Count')
ax.set_title('Error Counts vs Threshold')
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

---
## 12 — Final Evaluation with Optimized Threshold

In [ ]:
optimal_threshold = results_df.loc[opt_cost_idx, 'threshold']
y_final = (y_test_prob >= optimal_threshold).astype(int)

print(f"Final evaluation with optimized threshold = {optimal_threshold:.2f}")
print(f"{'='*55}")
print(f"\nROC-AUC:  {roc_auc_score(y_test, y_test_prob):.4f}")
print(f"PR-AUC:   {average_precision_score(y_test, y_test_prob):.4f}")
print(f"F1-Score: {f1_score(y_test, y_final):.4f}")
print(f"F2-Score: {fbeta_score(y_test, y_final, beta=2):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_final, target_names=['Legit', 'Fraud']))

tn, fp, fn, tp = confusion_matrix(y_test, y_final).ravel()
total_cost = (fn * COST_FN) + (fp * COST_FP) + (tp * COST_TP)
print(f"\nBusiness Impact:")
print(f"  Fraud caught:      {tp} / {tp+fn} ({100*tp/(tp+fn):.1f}%)")
print(f"  Legit blocked:     {fp} / {tn+fp} ({100*fp/(tn+fp):.2f}%)")
print(f"  Total error cost:  ${total_cost:,.0f}")

---
## 13 — GCP Deployment (Reference)

The cells below show GCP-specific operations as commented code. These require a GCP project with Vertex AI enabled.

In [ ]:
# ============================================================
# GCP DEPLOYMENT — Vertex AI
# Uncomment and configure with your project details to deploy.
# ============================================================

# --- Save model artifact ---
# best_model.save_model('model.bst')
# !gsutil cp model.bst gs://YOUR_BUCKET/models/fraud_detection/model.bst

# --- Upload model to Vertex AI ---
# from google.cloud import aiplatform
# aiplatform.init(project='YOUR_PROJECT', location='us-central1')
#
# model = aiplatform.Model.upload(
#     display_name='fraud-detection-xgboost',
#     artifact_uri='gs://YOUR_BUCKET/models/fraud_detection/',
#     serving_container_image_uri=(
#         'us-docker.pkg.dev/vertex-ai/prediction/xgboost-cpu.1-7:latest'
#     ),
# )

# --- Create endpoint and deploy ---
# endpoint = aiplatform.Endpoint.create(
#     display_name='fraud-detection-endpoint',
# )
#
# model.deploy(
#     endpoint=endpoint,
#     machine_type='n1-standard-4',
#     min_replica_count=2,     # Always-on (no cold start)
#     max_replica_count=10,    # Autoscale for peak traffic
#     traffic_percentage=100,
# )

# --- Online prediction ---
# prediction = endpoint.predict(instances=[feature_vector.tolist()])
# fraud_score = prediction.predictions[0]

# --- Model monitoring ---
# monitoring_job = aiplatform.ModelDeploymentMonitoringJob.create(
#     display_name='fraud-monitoring',
#     endpoint=endpoint,
#     logging_sampling_strategy={
#         'random_sample_config': {'sample_rate': 0.1}
#     },
#     schedule_config={'monitor_interval': {'seconds': 3600}},
# )

print("GCP deployment code shown above (commented).")
print("Uncomment and configure with your project to deploy.")

---
## 14 — Summary

### What We Built
1. **Synthetic data** — 10,000 transactions with realistic fraud patterns (0.5% fraud rate)
2. **Feature engineering** — amount ratios, velocity, time-based, geographic, interaction features
3. **Class imbalance** — compared SMOTE, class weights, and combined approaches
4. **XGBoost model** — trained with PR-AUC evaluation metric
5. **Threshold optimization** — cost-sensitive threshold selection
6. **GCP deployment path** — Vertex AI model upload, endpoint, monitoring

### GCP MLE Exam Connections
- **Feature Store** for online/offline feature consistency
- **Pub/Sub + Dataflow** for streaming feature computation
- **Vertex AI endpoints** with autoscaling for low-latency serving
- **Model monitoring** for concept drift detection
- **PR-AUC** as the correct metric for imbalanced classification

---
*CareerAlign — GCP Professional Machine Learning Engineer*